In [16]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

#Carga las herramientas que vamos a utilizar para trabajar con datos, MySQL y el archivo .env.

In [17]:
load_dotenv()
#Lee las credenciales y configuración de la base de datos guardadas en el archivo .env.

True

In [18]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
#Crea la variable engine.

In [19]:
query = """
SELECT *
FROM catalogo_productos;
"""

df_catalogo = pd.read_sql(query, engine)
#Hace una consulta SQL y carga todos los datos de catalogo_productos en un DataFrame de Pandas.

In [20]:
ventas_producto = (
    df_catalogo
    .groupby("product_id")
    .size()
    .reset_index(name="unidades_vendidas")
)
#Cuenta cuántas veces aparece cada producto para saber cuántas unidades se han vendido de cada uno.

In [21]:
precio_habitual = (
    df_catalogo
    .groupby(["product_id", "precio"])
    .size()
    .reset_index(name="veces")
    .sort_values(["product_id", "veces"], ascending=[True, False])
    .drop_duplicates("product_id")
)
#Cuenta cuántas veces aparece cada precio para cada producto y selecciona el precio que más se repite.

In [22]:
categoria_producto = (
    df_catalogo[["product_id", "categoria"]]
    .drop_duplicates("product_id")
)
#Selecciona product_id y categoria y deja una sola categoría por producto.

In [23]:

df_final = (
    precio_habitual[["product_id", "precio"]]
    .merge(categoria_producto, on="product_id", how="left")
    .merge(ventas_producto, on="product_id", how="left")
)
# Crear el DataFrame final


df_final["categoria"] = df_final["categoria"].fillna("sin_categoria")
# Sustituir las categorías vacías

In [24]:
total_unidades = df_final["unidades_vendidas"].sum()

print("Total de unidades vendidas:", total_unidades)
#Suma todas las unidades vendidas para obtener el total de ventas del conjunto de productos.

Total de unidades vendidas: 112650


In [25]:
import os

os.makedirs("data", exist_ok=True)

df_final.to_csv(
    "dt2_productos.csv",
    index=False,
    encoding="utf-8"
)

print("CSV exportado correctamente.")
#Crea la carpeta data si no existe y guarda el DataFrame final como archivo CSV.

CSV exportado correctamente.
